# Spatiotemporal Analysis of Urban Heat Island Effect in Multan, Pakistan (2013–2023)

Using Google Earth Engine, Machine Learning, Landsat 8 C2 L2 Imagery, and Remote Sensing Spectral Indices

Using Google Earth Engine, Machine Learning, Landsat 8 C2 L2 Imagery, and Remote Sensing Spectral Indices

Canonical extraction notebook. Original FYP: Musab Shahid, Bahauddin Zakariya University.

The November 2026 update removes fixed land-cover names from arbitrary cluster IDs and exports run metadata. Code cells pass local syntax checks; this version has not been rerun against Earth Engine. For an executed, account-free presentation of supplied CSVs, use `UHI_Results_From_Saved_Tables.ipynb`. The historical execution snapshot is retained separately. Its corresponding score cell parses; the clean notebook had a syntax error that is repaired in this release.

The later percentile proxy is a separate method, not a reconstruction of the original FYP vulnerability formula. Maps are optional.


In [ ]:
%pip -q install earthengine-api geemap pandas numpy matplotlib scipy

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from pathlib import Path

PROJECT_ID = "fyp-multan-spatiotemp-analysis"
YEARS = [2013, 2015, 2017, 2018, 2019, 2021, 2023]
SHOW_INTERACTIVE_MAPS = False
OUTPUT_DIR = Path("/content/multan_uhi_reconstructed_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCALE_M = 30

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=PROJECT_ID)
print("GEE initialized:", ee.Number(1).getInfo())
multan_point = ee.Geometry.Point([71.4753, 30.1575])
analysis_aoi = multan_point.buffer(25000)

if SHOW_INTERACTIVE_MAPS:
    m = geemap.Map()
    m.centerObject(multan_point, 10)
    m.addLayer(analysis_aoi, {"color": "cyan"}, "25-km AOI")
    display(m)

## 1. Source-derived Landsat 8 preprocessing

Dataset: `LANDSAT/LC08/C02/T1_L2`; 25-km radial AOI; May 1 to **exclusive September 30** of each selected year; metadata cloud cover below 10%. This retains the source code's exact date-filter semantics, including exclusion of September 30. Masking and scale factors follow the original PDF; derived indices use `normalizedDifference`, which can mask pixels with negative reflectance.

In [ ]:
def mask_landsat_l2(img):
    qa = img.select("QA_PIXEL")
    radsat = img.select("QA_RADSAT")
    mask = (qa.bitwiseAnd(1 << 0).eq(0)
            .And(qa.bitwiseAnd(1 << 1).eq(0))
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0))
            .And(radsat.eq(0)))
    return img.updateMask(mask)

def scale_landsat_l2(img):
    optical = img.select("SR_B[1-7]").multiply(0.0000275).add(-0.2)
    lst = img.select("ST_B10").multiply(0.00341802).add(149).subtract(273.15)
    return img.addBands(optical, overwrite=True).addBands(lst.rename("LST"), overwrite=True)

def add_indices(img):
    return img.addBands([
        img.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI"),
        img.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI"),
        img.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")
    ])

def process_year(year):
    collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                  .filterBounds(analysis_aoi)
                  .filterDate(f"{year}-05-01", f"{year}-09-30")
                  .filter(ee.Filter.lt("CLOUD_COVER", 10)))
    n = collection.size().getInfo()
    if not n:
        raise ValueError(f"No qualifying Landsat scenes for {year}")
    composite = (collection.map(mask_landsat_l2)
                 .map(scale_landsat_l2).map(add_indices)
                 .median().clip(analysis_aoi))
    return n, composite.select(["LST", "NDVI", "NDBI", "MNDWI"])

images, scene_counts = {}, {}
for year in YEARS:
    scene_counts[year], images[year] = process_year(year)
    print(year, "eligible scenes:", scene_counts[year])

## 2. Annual surface-temperature and index summaries

Spatial standard deviations describe within-AOI variability; they are **not confidence intervals**. Computations are made on per-year median composites.

In [ ]:
def reduce_year_summary(image, year):
    result = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=analysis_aoi, scale=SCALE_M,
        bestEffort=True, maxPixels=1e13).getInfo()
    return {"Year": year, **{k: np.nan if v is None else v for k, v in result.items()}}

summary_df = pd.DataFrame(
    [reduce_year_summary(images[y], y) for y in YEARS]
).set_index("Year")
summary_df = summary_df.reindex(columns=[
    "LST_mean", "LST_stdDev", "NDVI_mean", "NDVI_stdDev",
    "NDBI_mean", "NDBI_stdDev", "MNDWI_mean", "MNDWI_stdDev"
])
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / "annual_index_summary.csv")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, key, color in zip(axes.flat, ["LST", "NDVI", "NDBI", "MNDWI"],
                          ["firebrick", "forestgreen", "saddlebrown", "royalblue"]):
    mean, std = summary_df[key + "_mean"], summary_df[key + "_stdDev"]
    ax.plot(summary_df.index, mean, marker="o", color=color)
    ax.fill_between(summary_df.index, mean-std, mean+std, color=color, alpha=0.15)
    ax.set(xlabel="Year", ylabel="°C" if key=="LST" else "Index value", title=key)
    ax.grid(alpha=0.25)
fig.suptitle("Spatial AOI mean and standard deviation, selected years")
fig.tight_layout()
plt.show()

## 3. Optional interactive LST maps and 2013–2023 difference

Map widgets are optional because browser tile access can fail even when GEE is healthy. The simple LST difference does **not** standardize acquisition dates, cloud-observation coverage or weather.

In [ ]:
heat_vis = {"min": 25, "max": 50, "palette": ["blue","green","yellow","orange","red"]}
lst_difference = images[2023].select("LST").subtract(images[2013].select("LST"))
if SHOW_INTERACTIVE_MAPS:
    m = geemap.Map()
    m.centerObject(multan_point, 11)
    for year in YEARS:
        m.addLayer(images[year].select("LST"), heat_vis, f"LST {year}", shown=(year == 2023))
    display(m)
    d = geemap.Map()
    d.centerObject(multan_point, 11)
    d.addLayer(lst_difference, {"min":-5,"max":5,"palette":["blue","white","red"]},
               "2023 minus 2013 LST (°C)")
    display(d)
else:
    print("Map widgets skipped: SHOW_INTERACTIVE_MAPS = False")

## 4. Descriptive NDVI–LST regressions

As in the original PDF, the sample excludes NDVI ≤ 0; correlation is observational and affected by spatial autocorrelation, seasonal compositing and other confounding factors.

In [ ]:
def fit_ndvi_lst(image):
    samples = image.select(["LST", "NDVI"]).sample(
        region=analysis_aoi, scale=SCALE_M, numPixels=1000, seed=42
    ).getInfo()["features"]
    df = pd.DataFrame([f["properties"] for f in samples]).dropna()
    df = df.loc[df["NDVI"] > 0].copy()
    if len(df) < 3 or df.NDVI.nunique() < 2:
        raise ValueError("Insufficient valid sampled values")
    return df, stats.linregress(df.NDVI, df.LST)

regressions = {}
for year in YEARS:
    df, model = fit_ndvi_lst(images[year])
    regressions[year] = (df, model)
    print(year, "n", len(df), "R²", round(model.rvalue**2, 3))
df, model = regressions[2023]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df.NDVI, df.LST, s=12, alpha=.3)
xx = np.linspace(df.NDVI.min(), df.NDVI.max(), 100)
ax.plot(xx, model.intercept+model.slope*xx, c="red")
ax.set(xlabel="NDVI", ylabel="LST (°C)", title=f"2023 NDVI–LST, R²={model.rvalue**2:.3f}")
plt.show()

## 5. Exploratory 2023 clusters

The historical four-band K-Means method is retained without feature standardization. The output now uses neutral groups ordered by mean temperature; it no longer assigns Water/Vegetation/Barren/Urban from arbitrary IDs. These are exploratory spectral/temperature groups, not validated land-cover classes.


In [ ]:
stack_2023 = images[2023].select(["LST","NDVI","NDBI","MNDWI"])
training = stack_2023.sample(region=analysis_aoi, scale=SCALE_M,
                             numPixels=5000, seed=42)
clusterer = ee.Clusterer.wekaKMeans(4).train(training)
zones = stack_2023.cluster(clusterer)
rows = []
for cluster_id in range(4):
    mask = zones.eq(cluster_id)
    means = stack_2023.updateMask(mask).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=analysis_aoi,
        scale=SCALE_M, bestEffort=True, maxPixels=1e13).getInfo()
    area = ee.Image.pixelArea().updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=analysis_aoi,
        scale=SCALE_M, maxPixels=1e13).getInfo().get("area")
    rows.append({"cluster_id": cluster_id, **means, "area_m2": area})
zone_df = pd.DataFrame(rows).set_index("cluster_id")
zone_df["area_proportion"] = zone_df["area_m2"]/zone_df["area_m2"].sum()
# Numeric IDs are arbitrary; order descriptive groups by their observed profiles.
zone_df = zone_df.sort_values(["LST", "NDVI", "NDBI", "MNDWI"])
zone_df["temperature_rank"] = np.arange(1, len(zone_df) + 1)
zone_df["profile"] = zone_df["temperature_rank"].map(lambda i: f"Temperature group {i}")
display(zone_df)
zone_df.to_csv(OUTPUT_DIR/"kmeans_2023_cluster_summary.csv")
if SHOW_INTERACTIVE_MAPS:
    m = geemap.Map()
    m.centerObject(multan_point,11)
    m.addLayer(zones, {"min":0,"max":3,"palette":["blue","green","yellow","red"]},
               "Unsupervised clusters (unvalidated labels)")
    display(m)

## 6. NEW 2026 surface-vulnerability proxy — not the historical model

The original report mentions **high LST, low NDVI and high NDBI**, but the archived PDF contains no definitions for `vul_mask` and `vul_classes_combined`, no thresholds and no scoring formula.

This **new rule** uses the 2023 valid-pixel AOI distributions: LST ≥ 75th percentile, NDVI ≤ 25th percentile, NDBI ≥ 75th percentile. Score 0 = Low, scores 1–2 = Moderate, score 3 = Extreme. It measures a **relative satellite-surface proxy**, **not human heat vulnerability or health risk**. Do not calibrate it to the historical class proportions.

In [ ]:
source = images[2023].select(["LST","NDVI","NDBI"])
valid = source.mask().reduce(ee.Reducer.min())
source = source.updateMask(valid)
p = source.reduceRegion(
    reducer=ee.Reducer.percentile([25,75]), geometry=analysis_aoi,
    scale=SCALE_M, maxPixels=1e13, tileScale=4).getInfo()
keys = ("LST_p75","NDVI_p25","NDBI_p75")
if any(p.get(key) is None for key in keys):
    raise RuntimeError(f"Missing threshold statistics: {p}")
print("NEW threshold values:", {k:p[k] for k in keys})
lst_high = source.select("LST").gte(float(p["LST_p75"]))
ndvi_low = source.select("NDVI").lte(float(p["NDVI_p25"]))
ndbi_high = source.select("NDBI").gte(float(p["NDBI_p75"]))
score = lst_high.add(ndvi_low).add(ndbi_high)
new_classes = (
    score.eq(1).Or(score.eq(2)).multiply(1)
    .add(score.eq(3).multiply(2))
    .rename("vulnerability_class").toInt8().updateMask(valid)
)
if SHOW_INTERACTIVE_MAPS:
    m = geemap.Map()
    m.centerObject(multan_point,11)
    m.addLayer(new_classes, {"min":0,"max":2,
        "palette":["#2a9d8f","#f4a261","#d62828"]},
        "NEW relative vulnerability: Low/Moderate/Extreme")
    display(m)

In [ ]:
area_image = ee.Image.pixelArea().rename("area_m2").addBands(new_classes.rename("class"))
groups = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1,groupName="class"),
    geometry=analysis_aoi, scale=SCALE_M, maxPixels=1e13, tileScale=4
).getInfo().get("groups",[])
by_class = {int(x["class"]): float(x["sum"]) for x in groups}
total = sum(by_class.values())
if total <= 0:
    raise RuntimeError("No classified valid pixels")
names = {0:"Low",1:"Moderate",2:"Extreme"}
vulnerability_df = pd.DataFrame([
    {"class_code":i,"vulnerability":names[i],"area_km2":by_class.get(i,0)/1e6,
     "percent_of_valid_area":100*by_class.get(i,0)/total}
    for i in range(3)])
assert abs(vulnerability_df.percent_of_valid_area.sum()-100)<1e-6
display(vulnerability_df)
vulnerability_df.to_csv(OUTPUT_DIR/"new_vulnerability_2023_areas.csv",index=False)

## 7. Comparison with original report (not an equivalence test)

The FYP report lists **Low 38.157724%, Moderate 57.141238%, Extreme 4.701038%**. These came from an unrecovered model, not the new percentile rules. Differences document a change in method; they must not be interpreted as validation failures of the old method.

In [ ]:
historic = {"Low":38.157724, "Moderate":57.141238, "Extreme":4.701038}
comparison = vulnerability_df[["vulnerability","percent_of_valid_area"]].copy()
comparison["historical_report_percent"] = comparison.vulnerability.map(historic)
comparison["difference_percentage_points"] = (
    comparison.percent_of_valid_area-comparison.historical_report_percent)
display(comparison)
comparison.to_csv(OUTPUT_DIR/"historical_vs_new_comparison.csv",index=False)
print("CSV outputs saved to:", OUTPUT_DIR)

## Run provenance

The following cell exports environment, scene counts, thresholds and regression summaries for this run. Preserve these files with the CSVs. The local source was syntax checked, not independently rerun on Earth Engine; the offline results notebook has been executed from the supplied tables. Cloud masking, date selection and original pixel masks remain unchanged.


In [ ]:
import json, platform, importlib.metadata
run_metadata = {
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in ["earthengine-api", "geemap", "pandas", "numpy", "matplotlib", "scipy"]},
    "years": YEARS, "scene_counts": scene_counts,
    "dataset": "LANDSAT/LC08/C02/T1_L2",
    "season_start": "05-01", "season_end_exclusive": "09-30",
    "aoi_center_lon_lat": [71.4753, 30.1575], "radius_m": 25000,
    "nominal_scale_m": SCALE_M, "cloud_cover_lt": 10,
    "cluster_features_standardized": False,
    "proxy_thresholds": {k: float(p[k]) for k in keys},
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2))
regression_rows = []
for year, (sample_df, fitted) in regressions.items():
    sample_df.to_csv(OUTPUT_DIR / f"ndvi_lst_samples_{year}.csv", index=False)
    regression_rows.append({"year": year, "sample_n": len(sample_df), "slope": fitted.slope, "intercept": fitted.intercept, "r_squared": fitted.rvalue**2})
pd.DataFrame(regression_rows).to_csv(OUTPUT_DIR / "ndvi_lst_regression_summary.csv", index=False)
print("Run metadata and regression samples saved to", OUTPUT_DIR)
